# Chat With Multiple Documents (PDF, DOCX, TXT, PPTX) using AstraDB and LangChain

This notebook builds a Retrieval Augmented Generation (RAG) system that can read multiple documents of different formats and answer questions about their content.

The overall idea of RAG is simple. Instead of asking a language model to answer a question purely from what it memorized during training, we first search our own documents for the most relevant pieces of text, and then hand those pieces to the language model as context. The model then answers using that context, which makes the answers grounded in our real data instead of the model guessing.

This version of the notebook is written to run locally in VS Code (or any local Jupyter environment) instead of Google Colab. All the Google Colab specific pieces, such as reading secrets from `google.colab.userdata` and using `/content/` as the working folder, have been replaced with local equivalents such as a `.env` file and normal local folders.

The notebook is organized into clear linear steps:

1. Install the required Python packages
2. Import the libraries we will use
3. Load configuration and API keys from a local `.env` file
4. Create an embedding model, which turns text into numeric vectors
5. A quick demo of loading multiple PDFs using the Unstructured loader
6. A full example of loading PDFs one by one, storing them in AstraDB, and chatting with them
7. A full example of loading a folder containing mixed file types (PDF, DOCX, TXT, PPTX) using a Directory Loader, storing them in AstraDB, and chatting with them
8. A short conclusion summarizing what was built

Each code cell below has comments explaining what every important line does, so the notebook can be followed step by step even by someone new to LangChain or RAG.

## Step 1: Install the Required Packages

Before writing any code we need to install the Python packages this notebook depends on.

Because this notebook now runs locally instead of on Google Colab, a few extra tools are needed on your machine itself (not just Python packages):

* Poppler, which is required to read PDF files with the Unstructured loader
* Tesseract OCR, which is required if any PDF contains scanned images of text instead of selectable text

On Windows, the easiest way to install these two tools is with Conda, since your projects already use Conda virtual environments:

```
conda install -c conda-forge poppler
conda install -c conda-forge tesseract
```

If you are on macOS you can instead use Homebrew (`brew install poppler tesseract`), and on Ubuntu/Debian Linux you can use `sudo apt-get install poppler-utils tesseract-ocr`.

Run the cell below once to install all the Python packages needed for this notebook.

In [ ]:
# Install all Python packages required for this notebook in one place.
# Running this as a single line keeps it easy to copy into a local terminal on Windows as well.
%pip install langchain langchain-community langchain-openai langchain-astradb langchain-text-splitters openai datasets pypdf unstructured "unstructured[pdf]" "unstructured[pptx]" unstructured-pytesseract pdf2image pdfminer.six python-dotenv tiktoken Cython


### About Poppler and Tesseract

The Python package `unstructured-pytesseract` only gives Python access to Tesseract, it does not install the Tesseract program itself. The same is true for `pdf2image`, which needs the Poppler program to actually convert PDF pages into images.

Make sure you installed Poppler and Tesseract using Conda (or Homebrew/apt as shown above) before continuing, otherwise the PDF loading steps later in this notebook may fail.

## Step 2: Import the Libraries

Now that everything is installed, we import all the pieces we will use throughout the notebook.

Grouping every import together at the top of the notebook (instead of spreading them across many cells like in the original notebook) makes it much easier to see everything this project depends on at a glance.

In [1]:
# Standard library imports
import os

# python-dotenv lets us load secrets (API keys, tokens) from a local .env file
# instead of typing them directly into the notebook, which is important for
# keeping credentials out of any code we push to GitHub.
from dotenv import load_dotenv

# Hugging Face "datasets" library, kept here in case you want to load a
# ready made dataset instead of your own local documents.
from datasets import load_dataset

# Document loaders: these read files from disk and turn them into
# LangChain "Document" objects (basically text plus some metadata).
from langchain_community.document_loaders import (
    PyPDFLoader,       # reads a single PDF file, page by page
    UnstructuredPDFLoader,  # reads a single PDF using the Unstructured library
    DirectoryLoader,   # reads every supported file inside a folder
)

# The base Document class used across LangChain
from langchain_core.documents import Document

# Splits long text into smaller overlapping chunks, which is important
# because embedding models and language models work best on short passages.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The AstraDB vector store. This is where our document chunks and their
# embeddings will actually be stored and searched.
from langchain_astradb import AstraDBVectorStore

# Embedding model and chat model classes from OpenAI, wrapped for LangChain.
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Building blocks for creating a RAG "chain": a prompt template, a way to
# pass a question straight through unchanged, and a parser that converts
# the model's raw response into a plain string.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("All libraries imported successfully.")


f:\Multimodal-RAG-Systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8868\1476131862.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


All libraries imported successfully.


## Step 3: Load Astra DB Configuration from a Local `.env` File

The original notebook was written for Google Colab and loaded secrets using `google.colab.userdata.get(...)`. That approach only works inside Google Colab.

For a local VS Code setup, the recommended approach is to store sensitive credentials in a `.env` file located in the project folder. This keeps your API credentials separate from your code and prevents them from being accidentally committed to GitHub.

Since this notebook uses the free **Hugging Face** embedding model (`sentence-transformers/all-MiniLM-L6-v2`), an OpenAI API key is **not required**.

Before running the next cell, create a file named `.env` in the same folder as this notebook and add your Astra DB credentials:

```text
ASTRA_DB_API_ENDPOINT=your_astra_db_api_endpoint_here
ASTRA_DB_APPLICATION_TOKEN=your_astra_db_application_token_here
ASTRA_DB_KEYSPACE=default_keyspace
```

Also add `.env` to your `.gitignore` file so your credentials are never pushed to GitHub.


In [2]:
import os
from dotenv import load_dotenv

# Load all environment variables from the local .env file.
load_dotenv()

# Read the Astra DB credentials.
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

# Verify that all required credentials are available.
missing = [
    name for name, value in [
        ("ASTRA_DB_API_ENDPOINT", ASTRA_DB_API_ENDPOINT),
        ("ASTRA_DB_APPLICATION_TOKEN", ASTRA_DB_APPLICATION_TOKEN),
    ]
    if not value
]

if missing:
    raise ValueError(
        f"Missing required environment variables: {missing}. "
        "Please add them to your local .env file before continuing."
    )

print("Astra DB configuration loaded successfully.")

Astra DB configuration loaded successfully.


## Step 4: Create the Embedding Model

An embedding model converts a piece of text into a list of numbers (a vector) that captures the meaning of that text. Two pieces of text with similar meaning end up with vectors that are close together in that numeric space.

We will reuse this single embedding model everywhere in the notebook, both when we store document chunks in AstraDB and when we search for relevant chunks later.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

# Create one shared free Hugging Face embeddings object.
# The model runs locally and does not require an OpenAI API key.
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Free embedding model ready.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2336.22it/s]


Free embedding model ready.


## Step 5: Quick Demo, Loading Multiple PDFs with the Unstructured Loader

This section is a short, self-contained demo that shows how to load multiple PDF files from the current VS Code project folder, split their text into smaller chunks, generate embeddings, and store them in a local **in-memory Chroma vector store**.

The PDF files are located inside the following folder:

`f:\Multimodal-RAG-Systems\Multi_Document_Rag\data`

Each PDF file is loaded using `UnstructuredPDFLoader`. The extracted documents are then split into smaller chunks using `RecursiveCharacterTextSplitter`.

The shared free **Hugging Face embedding model** (`sentence-transformers/all-MiniLM-L6-v2`) created earlier is used to convert these chunks into vector embeddings. The embeddings and document chunks are stored temporarily in an **in-memory Chroma vector store**.

This section is intended as a quick demonstration of the basic RAG retrieval workflow. The in-memory vector store is temporary and is useful for experimentation and testing.

In **Steps 6 and 7**, we will replace this temporary in-memory setup with **Astra DB** to build a persistent vector storage and retrieval pipeline.


1. Set the PDF folder path

In [4]:
import os

# Local folder containing the PDF files.
pdf_folder_path = r"f:\Multimodal-RAG-Systems\Multi_Document_Rag\data"

# List all files to confirm that the path is correct.
print("Files found:", os.listdir(pdf_folder_path))

Files found: ['GenAI_Roadmap.pptx', 'got_book.txt', 'harry_potter_book.pdf', 'Lost in the Middle- How Language Models Use Long Contexts.pdf', 'Retrieval-Augmented-Generation-for-NLP.pdf']


2. Create a loader for each PDF

In [5]:
from langchain_community.document_loaders import UnstructuredPDFLoader

# Create one UnstructuredPDFLoader for each PDF file.
loaders = [
    UnstructuredPDFLoader(
        os.path.join(pdf_folder_path, file_name)
    )
    for file_name in os.listdir(pdf_folder_path)
    if file_name.lower().endswith(".pdf")
]

print(f"Created {len(loaders)} PDF loader(s).")

Created 3 PDF loader(s).


3. Load all PDF documents

In [6]:
# Load all PDF documents.
documents = []

for loader in loaders:
    documents.extend(loader.load())

print(f"Loaded {len(documents)} document(s).")

No languages specified, defaulting to English.
No languages specified, defaulting to English.
Cannot set non-stroke color: /'P0' is an invalid float value
Cannot set non-stroke color: /'P1' is an invalid float value
Cannot set non-stroke color because expected 3 components but got [/'P2']
Cannot set non-stroke color: /'P3' is an invalid float value
No languages specified, defaulting to English.


Loaded 3 document(s).


4. Split the documents into chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split the extracted PDF text into smaller chunks.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} text chunks.")

Created 390 text chunks.


5. Create the in-memory vector store

In [8]:
from langchain_chroma import Chroma

# Convert the document chunks into embeddings using the shared
# free Hugging Face embedding model created earlier.
#
# Store the embeddings and document chunks in an in-memory
# Chroma vector store.
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

print("In-memory vector store created from the PDF files.")

In-memory vector store created from the PDF files.


6. Create the retriever

In [9]:
# Create a retriever from the in-memory vector store.
# The retriever returns the top 3 most relevant chunks.
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("In-memory vector store retriever created successfully.")

In-memory vector store retriever created successfully.


7. Test the retriever with a question

In [10]:
query = "What is Multi-Document Question Answering?"

# Retrieve the most relevant document chunks.
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunks.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content)
    print("\nSource:", doc.metadata.get("source", "Unknown"))

Retrieved 3 relevant chunks.

--- Retrieved Chunk 1 ---
Table 1: Closed-book and oracle accuracy of language models on the multi-document question answering task.

2.3 Results and Discussion

We experiment with input contexts containing 10, 20, and 30 total documents. Figure 5 presents multi- document question answering performance when varying the position of relevant information within the input context. To contextualize model perfor- mance, we also evaluate on the closed-book and oracle settings (Table 1). In the closed-book setting, models are not given any documents in their input context, and must rely on their parametric memory to generate the correct answer. On the other hand, in the oracle setting, language models are given the single document that contains the answer and must use it to answer the question.

Source: f:\Multimodal-RAG-Systems\Multi_Document_Rag\data\Lost in the Middle- How Language Models Use Long Contexts.pdf

--- Retrieved Chunk 2 ---
2 Multi-Document Questio

### Step 5 Workflow

This Step 5 demonstrates **document retrieval using a local in-memory Chroma vector store**, not full question answering or answer generation with an LLM.

```text
PDF Files
    ↓
UnstructuredPDFLoader
    ↓
Documents
    ↓
RecursiveCharacterTextSplitter
    ↓
Text Chunks
    ↓
Hugging Face Embeddings
    ↓
In-Memory Chroma Vector Store
    ↓
Retriever
    ↓
Relevant Document Chunks
```

**Flow:** PDF Files → `UnstructuredPDFLoader` → Documents → Text Splitter → Text Chunks → Hugging Face Embeddings → In-Memory Chroma Vector Store → Retriever → Relevant Document Chunks


## Step 6: Loading Multiple PDFs with PyPDFLoader and Storing Them in Astra DB

In **Step 5**, we created a temporary **in-memory Chroma vector store** to demonstrate the basic document retrieval workflow. The vector store exists only for the current session and is not intended to be used as the persistent vector database for this project.

In this step, we build a more persistent RAG ingestion pipeline using **Astra DB** as the vector database.

We will load multiple PDF documents using `PyPDFLoader`, split the extracted text into smaller overlapping chunks using `RecursiveCharacterTextSplitter`, generate vector embeddings using the shared free **Hugging Face embedding model**, and store the document chunks and their embeddings in **Astra DB**.

### Step 6 Workflow

```text id="rlqxzj"
PDF Files
    ↓
PyPDFLoader
    ↓
Documents
    ↓
RecursiveCharacterTextSplitter
    ↓
Text Chunks
    ↓
Hugging Face Embeddings
    ↓
Astra DB Vector Store
    ↓
Retriever
    ↓
Relevant Document Chunks
```

Unlike the temporary Chroma setup used in Step 5, Astra DB provides persistent vector storage, allowing the stored document embeddings to be retrieved and reused later without rebuilding the vector database every time.


1. Set the PDF folder path

In [11]:
import os

# Local folder containing the PDF files for this section.
pdf_folder_path = r"f:\Multimodal-RAG-Systems\Multi_Document_Rag\data"

# Get only PDF files from the folder.
pdf_files = [
    file_name
    for file_name in os.listdir(pdf_folder_path)
    if file_name.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF file(s):")
print(pdf_files)

Found 3 PDF file(s):
['harry_potter_book.pdf', 'Lost in the Middle- How Language Models Use Long Contexts.pdf', 'Retrieval-Augmented-Generation-for-NLP.pdf']


2. Create the text splitter

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter to break long PDF documents
# into smaller overlapping chunks.
#
# chunk_size:
# Maximum number of characters in each chunk.
#
# chunk_overlap:
# Number of characters shared between consecutive chunks
# to help preserve context across chunk boundaries.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64
)

print("Text splitter created successfully.")

Text splitter created successfully.


3. Load and split all PDF files

In [13]:
from langchain_community.document_loaders import PyPDFLoader

# Load each PDF file and split its content into smaller chunks.
# All chunks from all PDFs are collected into one list.
docs_from_pdf = []

for file_name in pdf_files:
    file_path = os.path.join(pdf_folder_path, file_name)

    # Create a loader for the current PDF.
    loader = PyPDFLoader(file_path)

    # Load the PDF and split it into chunks.
    chunks = loader.load_and_split(
        text_splitter=splitter
    )

    # Add the chunks to the combined document list.
    docs_from_pdf.extend(chunks)

print(f"Created {len(docs_from_pdf)} document chunks from {len(pdf_files)} PDF file(s).")

Created 640 document chunks from 3 PDF file(s).


4. Connect to the Astra DB Vector Store

In [14]:
from langchain_astradb import AstraDBVectorStore

# Connect to (or create) a collection in Astra DB.
#
# The collection stores:
# 1. Document text chunks
# 2. Vector embeddings generated by the Hugging Face embedding model
# 3. Document metadata, such as source and page number
#
# These vectors can later be searched using semantic similarity.
vstore = AstraDBVectorStore(
    embedding=embedding,
    collection_name="multipdf_vector",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
)

print("Connected to Astra DB collection: multidoc_vector")

Connected to Astra DB collection: multidoc_vector


5. Store the PDF Chunks in Astra DB

In [15]:
# Add all PDF chunks to the Astra DB vector store.
#
# For each document chunk:
# 1. The Hugging Face embedding model converts the text into a vector
# 2. The text chunk, vector, and metadata are stored in Astra DB
#
# Astra DB returns a unique ID for each stored document chunk.
inserted_ids_from_pdf = vstore.add_documents(
    docs_from_pdf
)

print(
    f"Inserted {len(inserted_ids_from_pdf)} "
    "document chunks into Astra DB."
)

Inserted 640 document chunks into Astra DB.


6. Create the Astra DB Retriever

In [16]:
# Convert the Astra DB vector store into a retriever.
#
# When a query is provided:
# 1. The query is converted into an embedding
# 2. Astra DB performs a vector similarity search
# 3. The top 3 most relevant document chunks are returned
retriever = vstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Astra DB retriever ready.")

Astra DB retriever ready.


7. Test the Retriever

In [17]:
# Ask a test question.
query = "What is Retrieval Fusions?"

# Retrieve the top 3 most relevant document chunks from Astra DB.
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunks.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content)
    print("\nSource:", doc.metadata.get("source", "Unknown"))

Retrieved 3 relevant chunks.

--- Retrieved Chunk 1 ---
retrievals, optimizations for post-processing are applied (line 4). The time complexity
isO(C encode +C ann +k·C IO +C post), whereC ann,C IO, andC post are the cost of
performing ANN search, fetching neighbor values from the disk, and post-processing.
The space complexity isO(k·d).
4 Retrieval Fusions
Retrieval fusions refer to how to leverage the retrieved information to improve gen-
erators’ performance. Basically, there are four types of retrieval fusions: query-based

Source: f:\Multimodal-RAG-Systems\Multi_Document_Rag\data\Retrieval-Augmented-Generation-for-NLP.pdf

--- Retrieved Chunk 2 ---
et al. 2022), chunks (Borgeaud et al. 2022; Wang et al. 2023a), documents (Cheng
et al. 2023a). For the fusion in the logits, most works combine the logits of retrievals
and inputs by ensemble techniques (Shi et al. 2024; Guu et al. 2020; Lewis et al. 2020;
Mueller et al. 2023).
Instead of designing different knowledge fusions for QA sy

In [ ]:
# Ask a test question from 2nd pdf
query = "What is Skopostheorie?"

# Retrieve the top 3 most relevant document chunks from Astra DB.
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunks.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content)
    print("\nSource:", doc.metadata.get("source", "Unknown"))

Retrieved 3 relevant chunks.

--- Retrieved Chunk 1 ---
that the application of reduplicative is not good to help the 
reader get the immediate understanding of the spell effect. 
They have to refer to the whole story to judge what the 
spell has done to the target.
3.  COMPARISON OF TWO VERSIONS 
FROM THE PERSPECTIVE OF  
SKOPOSTHEORIE
From Skopostheories, the purpose of translation to some 
extent determines the outcome of translation. In the content 
below the differences between the spell translations of

Source: f:\Multimodal-RAG-Systems\Multi_Document_Rag\data\harry_potter_book.pdf

--- Retrieved Chunk 2 ---
means the receiver should be able to understand it; it should 
make sense in the communicative situation and culture in 
which it is received (Nord, 2001, p.32). 
In a word, there are no such things as right or wrong, 
faithfulness or unfaithfulness within skopostheorie, 
and the translation Skopos determines the translation 
process. Skopostheorie accounts for different stra

In [20]:
# Ask a test question from 3rd pdf
query = "What is Multi-Document Question Answering?"

# Retrieve the top 3 most relevant document chunks from Astra DB.
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunks.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(doc.page_content)
    print("\nSource:", doc.metadata.get("source", "Unknown"))

Retrieved 3 relevant chunks.

--- Retrieved Chunk 1 ---
model performance on multi-document question
answering, which requires models to find relevant
information within an input context and use it to
answer the question. In particular, we make con-
trolled changes to the length of the input context
and the position of the relevant information and
measure changes in task performance.
2.1 Experimental Setup
In the multi-document question answering task, the
model inputs are (i) a question to answer and (ii) k
documents (e.g., passages from Wikipedia), where

Source: f:\Multimodal-RAG-Systems\Multi_Document_Rag\data\Lost in the Middle- How Language Models Use Long Contexts.pdf

--- Retrieved Chunk 2 ---
alization minimally affects performance trends in
the multi-document question answering task (Fig-
ure 9); it slightly improves performance when the
relevant information is located at the very begin-
ning of the input context, but slightly decreases
performance in other settings.
4.3 Effe

9. Build the Prompt Template

The prompt template defines the instructions sent to the language model together with the retrieved document chunks and the user's question.

For every user query, the retriever first searches Astra DB and returns the most relevant document chunks. These retrieved chunks replace the {context} placeholder, while the user's query replaces the {question} placeholder.

The completed prompt is then passed to the language model, which generates an answer based only on the retrieved context. If the answer is not present in the retrieved documents, the model should clearly state that it cannot find the information instead of making up an answer.

In [21]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt template for Retrieval-Augmented Generation (RAG).
prompt_text = """
You are a helpful AI assistant.

Answer the user's question using only the information provided in the retrieved context.

If the answer cannot be found in the context, simply say:
"I couldn't find the answer in the provided documents."

Keep your answer clear, concise, and accurate.

Context:
{context}

Question:
{question}

Answer:
"""

rag_prompt = ChatPromptTemplate.from_template(prompt_text)

print("Prompt template created successfully.")

Prompt template created successfully.


9. Load the API key

In [22]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in the .env file.")

print("Groq API key loaded successfully.")

Groq API key loaded successfully.


10. Create the Groq LLM

In [23]:
from langchain_groq import ChatGroq

# Initialize the Groq chat model.
# This model generates the final answer using the
# retrieved context from Astra DB.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY,
)

print("Groq LLM ready.")

Groq LLM ready.


11. Build the RAG Chain

In [24]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")

RAG chain built successfully.


12. Ask a Question

In [25]:
query = "What is Advanced ANN Indexing?"

response = rag_chain.invoke(query)

print(response)

Advanced ANN Indexing refers to the methods or structures used to organize and manage data so that the approximate-nearest-neighbor search process is optimized for retrieval quality and retrieval efficiency.


In [27]:
query = "According to Skopostheorie, what is more important than literal translation?"

response = rag_chain.invoke(query)

print(response)

According to Skopostheorie, the viability of the translation brief depends on the circumstances of the target culture, and the target text should conform to the standards of “intratextual coherence”, which means the receiver should be able to understand it. This implies that making sense in the communicative situation and culture is more important than literal translation.


## Step 7: Loading a Folder of Mixed Document Types and Storing Them in Astra DB

In **Step 6**, we built a Retrieval-Augmented Generation (RAG) pipeline using **PDF** documents only.

In this step, we expand the knowledge base by loading an entire folder containing multiple document formats such as **PDF**, **DOCX**, **TXT**, and **PPTX**.

Instead of manually selecting a loader for each file type, we use **`DirectoryLoader`**, which automatically detects the appropriate loader based on each file's extension.

After loading the documents, we split them into smaller overlapping chunks using `RecursiveCharacterTextSplitter`, generate embeddings using the shared **Hugging Face embedding model**, and store the chunks in a separate **Astra DB collection** dedicated to the mixed-document knowledge base.


Using a separate Astra DB collection keeps the mixed-document knowledge base independent from the PDF-only collection created in **Step 6**.

### Step 7 Workflow

```text
Mixed Documents
(PDF, DOCX, TXT, PPTX)
            ↓
      DirectoryLoader
            ↓
        Documents
            ↓
RecursiveCharacterTextSplitter
            ↓
      Document Chunks
            ↓
Hugging Face Embeddings
            ↓
 Astra DB Vector Store
            ↓
       Retriever
            ↓
 Relevant Document Chunks
```


1. Set the document folder

In [28]:
import os
import shutil

# Folder containing mixed document types.
docs_folder_path = r"f:\Multimodal-RAG-Systems\Multi_Document_Rag\data"

# Remove the hidden Jupyter checkpoint folder if it exists.
checkpoints_path = os.path.join(
    docs_folder_path,
    ".ipynb_checkpoints"
)

if os.path.isdir(checkpoints_path):
    shutil.rmtree(checkpoints_path)
    print("Removed .ipynb_checkpoints folder.")
else:
    print("No .ipynb_checkpoints folder found.")

No .ipynb_checkpoints folder found.


2. Load and split the documents

In [50]:
import os

for file in os.listdir(docs_folder_path):
    print(file)

GenAI_Roadmap.pptx
harry_potter_book.pdf
Lost in the Middle- How Language Models Use Long Contexts.pdf
Retrieval-Augmented-Generation-for-NLP.pdf
state_of_the_union.txt


In [51]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Automatically load all supported document types.
loader = DirectoryLoader(
    docs_folder_path
)

# Reuse the same chunking strategy from Step 6.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64
)

# Load and split every document.
docs = loader.load_and_split(
    text_splitter=splitter
)

print(f"Loaded {len(docs)} document chunks.")

No languages specified, defaulting to English.
No languages specified, defaulting to English.
Cannot set non-stroke color: /'P0' is an invalid float value
Cannot set non-stroke color: /'P1' is an invalid float value
Cannot set non-stroke color because expected 3 components but got [/'P2']
Cannot set non-stroke color: /'P3' is an invalid float value
No languages specified, defaulting to English.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


Loaded 881 document chunks.


Check how many files were loaded

In [53]:
unique_files = sorted({doc.metadata["source"] for doc in docs})

print(f"Total files loaded: {len(unique_files)}\n")

for i, file in enumerate(unique_files, 1):
    print(f"{i}. {file}")

Total files loaded: 5

1. F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\GenAI_Roadmap.pptx
2. F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\Lost in the Middle- How Language Models Use Long Contexts.pdf
3. F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\Retrieval-Augmented-Generation-for-NLP.pdf
4. F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\harry_potter_book.pdf
5. F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\state_of_the_union.txt


3. Create a new Astra DB vector store

In [55]:
from langchain_astradb import AstraDBVectorStore

# Create a separate Astra DB collection for the
# mixed-document knowledge base.
vstore_multidoc = AstraDBVectorStore(
    embedding=embedding,
    collection_name="multidoc_vector",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
)

print("Connected to Astra DB collection: multidocument_vector")

Connected to Astra DB collection: multidocument_vector


4. Store the document chunks

In [56]:
# Generate embeddings for every document chunk and
# store them in Astra DB.
inserted_ids = vstore_multidoc.add_documents(
    docs
)

print(
    f"Inserted {len(inserted_ids)} "
    "document chunks into Astra DB."
)

Inserted 881 document chunks into Astra DB.


5. Create the retriever

In [57]:
# Create a retriever for the mixed-document collection.
retriever_multidoc = vstore_multidoc.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully.")

Retriever created successfully.


6. Build the Prompt Template for the Multi-Document Knowledge Base

The prompt template defines how the language model should answer questions using the retrieved context from the multi-document knowledge base.

For every user question, the retriever searches the Astra DB collection and returns the most relevant document chunks. These chunks are inserted into the `{context}` placeholder, while the user's query replaces the `{question}` placeholder.

The language model should generate answers only from the retrieved context. If the required information is not available, it should clearly state that the answer could not be found in the provided documents.

In [58]:
roadmap_prompt_text = """                                
You are a helpful AI assistant.

Answer the user's question using only the information provided in the retrieved context.

If the answer cannot be found in the context, simply say:

"I couldn't find the answer in the provided documents."

Keep your answer clear, concise, and accurate.

Context:
{context}

Question:
{question}

Answer:
"""

7. Build the RAG chain

In [59]:
# Reuse the same chat model instance created earlier in Step 6.
# Build the second RAG chain, following the same pattern as before but
# using this section's retriever and prompt.
multidoc_chain = (
    {
        "context": retriever_multidoc,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")


RAG chain built successfully.


8. Ask a Question

Example 1: Querying Information from a PDF Document

In [60]:
query1 = "What are the stages for using the retriever?"

# Retrieve the relevant document chunks
retrieved_docs = retriever_multidoc.invoke(query1)

print(f"Retrieved {len(retrieved_docs)} document(s).\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"========== Document {i} ==========")
    print("Source :", doc.metadata.get("source", "Unknown"))
    print("Page   :", doc.metadata.get("page", "N/A"))
    print("\nContent:\n")
    print(doc.page_content)
    print("\n" + "=" * 80 + "\n")

Retrieved 3 document(s).

========== Document 1 ==========
Source : F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\Retrieval-Augmented-Generation-for-NLP.pdf
Page   : N/A

Content:

3 Retriever

Figure 2 shows the two stages for using the retriever, which involve first building the retriever and then querying the retriever. The following sections will introduce details about each stage.

3.1 Building the Retriever


========== Document 2 ==========
Source : F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\Retrieval-Augmented-Generation-for-NLP.pdf
Page   : N/A

Content:

Fig. 2 Two stages of using the retriever.

3.1.1 Chunking Corpus


========== Document 3 ==========
Source : F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\Retrieval-Augmented-Generation-for-NLP.pdf
Page   : N/A

Content:

3.2 Querying the Retriever

This section will explain how to query the pre-built retriever, which basically includes three steps as shown in Figure 2(b): encoding queries, ANN search, and post

Example 2: Querying Information from a PowerPoint (PPTX) Document

In [61]:
query2 = "Give the Roadmap of GenAI?"

# Retrieve the relevant document chunks
retrieved_docs = retriever_multidoc.invoke(query2)

print(f"Retrieved {len(retrieved_docs)} document(s).\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"========== Document {i} ==========")
    print("Source :", doc.metadata.get("source", "Unknown"))
    print("Page   :", doc.metadata.get("page", "N/A"))
    print("\nContent:\n")
    print(doc.page_content)
    print("\n" + "=" * 80 + "\n")

Retrieved 3 document(s).

========== Document 1 ==========
Source : F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\GenAI_Roadmap.pptx
Page   : N/A

Content:

GENERATIVE AI

The Generative AI Roadmap

A phase by phase learning path from foundations to autonomous AI agents

Foundations

Deep Learning

LLMs

Prompting

RAG

Fine-tuning

Agents

Deployment

Prepared as a self study and team onboarding guide



The Roadmap at a Glance

Eight phases, moving from core fundamentals to production ready AI systems

01

03

05

07

Foundations

Large Language

Models

Retrieval

Augmented Gen.

AI Agents &

Tool Use

02

04

06

08

Deep Learning

& NLP

Prompt


========== Document 2 ==========
Source : F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\GenAI_Roadmap.pptx
Page   : N/A

Content:

Text representation

Tokenization, word and sentence embeddings, and how raw text becomes numerical input for a model.

COMMON TOOLS

Core NLP tasks

PyTorch, TensorFlow, Hugging Face datasets

Classif

Example 3: Querying Information from a Text (TXT) Document

In [62]:
query3 = "Why does Gared want to turn back?"

# Retrieve the relevant document chunks
retrieved_docs = retriever_multidoc.invoke(query3)

print(f"Retrieved {len(retrieved_docs)} document(s).\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"========== Document {i} ==========")
    print("Source :", doc.metadata.get("source", "Unknown"))
    print("Page   :", doc.metadata.get("page", "N/A"))
    print("\nContent:\n")
    print(doc.page_content)
    print("\n" + "=" * 80 + "\n")

Retrieved 3 document(s).

========== Document 1 ==========
Source : F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\state_of_the_union.txt
Page   : N/A

Content:

Heath’s widow Danielle is here with us tonight. They loved going to Ohio State football games. He loved building Legos with their daughter.

But cancer from prolonged exposure to burn pits ravaged Heath’s lungs and body.

Danielle says Heath was a fighter to the very end.

He didn’t know how to stop fighting, and neither did she.

Through her pain she found purpose to demand we do better.

Tonight, Danielle—we are.


========== Document 2 ==========
Source : F:\Multimodal-RAG-Systems\Multi_Document_Rag\data\state_of_the_union.txt
Page   : N/A

Content:

A former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. Since she’s been nominated, she’s received a broad range of support—from the Fraternal Order of Police to form

## Conclusion

In this project, we built a complete **Multi-Document Retrieval-Augmented Generation (RAG)** system capable of answering questions from multiple document formats, including **PDF, DOCX, TXT, and PPTX** files.

Instead of limiting the knowledge base to a single document type, we used **DirectoryLoader** to automatically load different file formats from a local folder. The documents were split into smaller chunks, converted into vector embeddings using a free **Hugging Face embedding model**, and stored in **Astra DB**, a cloud-native vector database.

When a user submits a question, the retriever searches Astra DB for the most relevant document chunks. These retrieved chunks are combined with the user's query through a prompt template and passed to a **Groq-hosted Llama 3.3 70B** language model, which generates an accurate answer based only on the retrieved context.

The project demonstrates how a modern RAG pipeline can efficiently build a searchable knowledge base from multiple document types while reducing hallucinations by grounding responses in the original source documents.

Overall, this project provides a scalable and production-ready foundation for building AI-powered document assistants, enterprise knowledge bases, research assistants, internal search systems, and question-answering applications.


## Overall Project Workflow

```text
                Mixed Documents Folder
       (PDF, DOCX, TXT, PPTX Files)
                        │
                        ▼
               DirectoryLoader
        Automatically loads all supported
               document file formats
                        │
                        ▼
                  Documents Loaded
                        │
                        ▼
     RecursiveCharacterTextSplitter
        Splits documents into chunks
                        │
                        ▼
       Hugging Face Embedding Model
 (sentence-transformers/all-MiniLM-L6-v2)
        Converts chunks into vectors
                        │
                        ▼
              Astra DB Vector Store
      Stores document embeddings securely
                        │
                        ▼
             Astra DB Retriever
 Finds the most relevant document chunks
        using semantic similarity search
                        │
                        ▼
              ChatPromptTemplate
 Combines retrieved context with the user
                 question
                        │
                        ▼
      Groq Llama 3.3 70B Language Model
      Generates an answer grounded in the
             retrieved document context
                        │
                        ▼
              Final AI Response
        Accurate answer with reduced
             hallucination risk
```


## Tools and Technologies Used

| Category                    | Tools / Technologies                                    |
| --------------------------- | ------------------------------------------------------- |
| **Programming Language**    | Python                                                  |
| **Development Environment** | Visual Studio Code, Jupyter Notebook                    |
| **Document Loaders**        | DirectoryLoader, PyPDFLoader                            |
| **Supported File Types**    | PDF, DOCX, TXT, PPTX                                    |
| **Text Splitting**          | RecursiveCharacterTextSplitter                          |
| **Embedding Model**         | Hugging Face (`sentence-transformers/all-MiniLM-L6-v2`) |
| **Vector Database**         | Astra DB (DataStax)                                     |
| **Retriever**               | Astra DB Retriever                                      |
| **LLM**                     | Groq – Llama 3.3 70B Versatile                          |
| **Prompt Engineering**      | ChatPromptTemplate                                      |
| **RAG Framework**           | LangChain                                               |
| **Output Parsing**          | StrOutputParser                                         |
| **Environment Management**  | python-dotenv (.env)                                    |
| **Vector Search**           | Semantic Similarity Search                              |
| **Architecture**            | Retrieval-Augmented Generation (RAG)                    |
